In [ ]:
def check_stationarity(series, name):
    result = adfuller(series.dropna())
    pval = result[1]
    print(f"{name:25s} | ADF p-value: {pval:.4f} | {'Stationary ✓' if pval < 0.05 else 'Non-stationary ✗'}")

def build_lag_matrix(df, target, max_lag=4):
    X = pd.DataFrame()
    for col in df.columns:
        if col == target:
            continue
        for lag in range(2, max_lag + 1):   # lag 2+ only (predictive model)
            X[f'{col}_L{lag}'] = df[col].shift(lag)
    y = df[target]
    X = X.dropna()
    y = y[y.index.isin(X.index)]
    return X, y

def run_lag_analysis(df, country):
    """
    Runs stationarity checks, transforms, and LassoCV lag selection
    for a given country dataframe. Returns surviving lag coefficients.
    """
    credit_col = f'{country}_Credit'

    # ── column sets ───────────────────────────────────────────────────────────
    log_diff_cols = [c for c in ['rGDP', 'house_prices', 'OilPrice', 'ExchangeRate', 'Open',
                                  'UK_rGDP', 'US_rGDP',
                                  'UK_house_prices', 'US_house_prices',
                                  'UK_OilPrice', 'US_OilPrice',
                                  'UK_ExchangeRate', 'US_ExchangeRate',
                                  'UK_FTSE_Open', 'US_SP500_Open'] if c in df.columns]

    first_diff_cols = [c for c in df.columns
                       if c not in log_diff_cols and c != credit_col]

    print(f"\n{'='*60}")
    print(f"  {country} — Stationarity Checks")
    print(f"{'='*60}")
    for col in df.columns:
        check_stationarity(df[col], col)

    # ── transform ─────────────────────────────────────────────────────────────
    df_transformed = pd.DataFrame()

    for col in log_diff_cols:
        df_transformed[f'd_{col}'] = np.log(df[col]).diff()

    for col in first_diff_cols:
        df_transformed[f'd_{col}'] = df[col].diff()

    # credit target — first difference
    target_name = f'd_{credit_col}'
    df_transformed[target_name] = df[credit_col].diff()

    df_transformed.dropna(inplace=True)

    # ── LassoCV ───────────────────────────────────────────────────────────────
    X, y = build_lag_matrix(df_transformed, target_name, max_lag=4)

    lasso = LassoCV(cv=5, max_iter=10000)
    lasso.fit(X, y)

    important = pd.Series(lasso.coef_, index=X.columns)
    surviving = important[important != 0].sort_values()

    print(f"\n  {country} — Surviving Lags (L2+) for {target_name}:")
    print(surviving if not surviving.empty else "  None survived regularisation.")

    return surviving

# ── run ───────────────────────────────────────────────────────────────────────

surviving_UK = run_lag_analysis(merged_UK, 'UK')
surviving_US = run_lag_analysis(merged_US, 'US')

# **Adding Lag Variables for Regression Models:**
______

# **Creating Dataset**
___